# Assignment One
## Computer Vision - Image Processing

This notebook covers:
1. Adding Gaussian noise to an image
2. Manual convolution filtering using NumPy
3. Gaussian denoising and sharpening via convolution
4. Template matching using convolution and correlation

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

---
## Question 1: Adding Gaussian Noise to an Image

We load the dog image and add Gaussian noise with **mean=0** and **std=15**.

In [ ]:
# Load the dog image
dog_img = Image.open('dog.jpg').convert('L')  # Convert to grayscale
dog_array = np.array(dog_img, dtype=np.float64)

print(f'Image shape: {dog_array.shape}')
print(f'Pixel value range: [{dog_array.min():.1f}, {dog_array.max():.1f}]')

In [ ]:
def add_gaussian_noise(image, mean=0, std=15, seed=42):
    """
    Add Gaussian noise to an image.
    
    Parameters:
        image : np.ndarray  - Input image (float64)
        mean  : float       - Mean of the Gaussian distribution
        std   : float       - Standard deviation of the Gaussian distribution
        seed  : int         - Random seed for reproducibility
    Returns:
        noisy_image : np.ndarray - Noisy image clipped to [0, 255]
        noise       : np.ndarray - The noise array that was added
    """
    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=mean, scale=std, size=image.shape)
    noisy_image = image + noise
    # Clip values to valid pixel range
    noisy_image = np.clip(noisy_image, 0, 255)
    return noisy_image, noise

# Apply Gaussian noise
noisy_dog, noise = add_gaussian_noise(dog_array, mean=0, std=15)

print(f'Noise statistics -> Mean: {noise.mean():.4f}, Std: {noise.std():.4f}')

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(dog_array, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original Image', fontsize=13)
axes[0].axis('off')

axes[1].imshow(noise, cmap='gray')
axes[1].set_title(f'Gaussian Noise\n(mean=0, std=15)', fontsize=13)
axes[1].axis('off')

axes[2].imshow(noisy_dog, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('Noisy Image', fontsize=13)
axes[2].axis('off')

plt.suptitle('Question 1: Adding Gaussian Noise', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('q1_gaussian_noise.png', dpi=150, bbox_inches='tight')
plt.show()
print('Q1 figure saved.')

---
## Question 2: Manual Convolution using NumPy

Implement convolution from scratch (no `scipy.signal.convolve` or similar):
1. Zero-pad the image
2. Flip the kernel both horizontally and vertically
3. Compute the weighted sum at each pixel position

In [ ]:
def convolve2d(image, kernel):
    """
    Perform 2D convolution manually using NumPy.
    
    Steps:
      i)  Zero-pad the image so output has the same spatial dimensions.
      ii) Flip the kernel both horizontally and vertically (180-degree rotation).
      iii)Slide the flipped kernel over the padded image and compute weighted sums.
    
    Parameters:
        image  : np.ndarray (H, W)       - Grayscale input image
        kernel : np.ndarray (kH, kW)     - Convolution kernel
    Returns:
        output : np.ndarray (H, W)       - Convolved image (same size as input)
    """
    img_h, img_w = image.shape
    k_h, k_w = kernel.shape

    # --- Step i: Zero-padding ---
    pad_h = k_h // 2
    pad_w = k_w // 2
    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0)

    # --- Step ii: Flip the kernel (180-degree rotation) ---
    flipped_kernel = np.flip(kernel)   # equivalent to np.flip(kernel, axis=(0,1))

    # --- Step iii: Compute weighted sums (slide the flipped kernel) ---
    output = np.zeros_like(image, dtype=np.float64)
    for i in range(img_h):
        for j in range(img_w):
            region = padded[i:i + k_h, j:j + k_w]
            output[i, j] = np.sum(region * flipped_kernel)

    return output


# Define the kernel from the assignment
kernel = np.array([[1, 0, -1],
                   [2, 0, -2],
                   [1, 0, -1]], dtype=np.float64)

print('Kernel:')
print(kernel)
print()
print('Flipped kernel (used in convolution):')
print(np.flip(kernel))

In [ ]:
# Apply the convolution to the original dog image
print('Running convolution (this may take a moment for large images)...')
conv_result = convolve2d(dog_array, kernel)
print('Done!')

# Normalise to [0,255] for display
conv_display = conv_result - conv_result.min()
conv_display = (conv_display / conv_display.max() * 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(dog_array, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original Image', fontsize=13)
axes[0].axis('off')

axes[1].imshow(conv_display, cmap='gray')
axes[1].set_title('After Convolution\n(Sobel-like edge detection)', fontsize=13)
axes[1].axis('off')

plt.suptitle('Question 2: Manual Convolution', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('q2_convolution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Q2 figure saved.')

---
## Question 3: Gaussian Denoising & Image Sharpening

### Part A – Remove noise using a 7×7 Gaussian filter (σ=1.0)
### Part B – Sharpen the image using the provided sharpening kernel

In [ ]:
def make_gaussian_kernel(size, sigma):
    """
    Generate a 2D Gaussian kernel.
    
    Parameters:
        size  : int   - Kernel size (must be odd)
        sigma : float - Standard deviation of the Gaussian
    Returns:
        kernel : np.ndarray (size, size) - Normalised Gaussian kernel
    """
    assert size % 2 == 1, 'Kernel size must be odd.'
    half = size // 2
    # Create coordinate grids
    y, x = np.mgrid[-half:half+1, -half:half+1]
    # Gaussian formula
    gaussian = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    # Normalise so the kernel sums to 1
    kernel = gaussian / gaussian.sum()
    return kernel


# Build the 7x7 Gaussian kernel with sigma=1.0
gauss_kernel = make_gaussian_kernel(size=7, sigma=1.0)

print('7x7 Gaussian kernel (sigma=1.0):')
np.set_printoptions(precision=6, suppress=True)
print(gauss_kernel)
print(f'\nKernel sum (should be ~1.0): {gauss_kernel.sum():.6f}')

In [ ]:
# Apply Gaussian filter to the NOISY image (from Q1) to denoise it
print('Denoising with 7x7 Gaussian filter...')
denoised_dog = convolve2d(noisy_dog, gauss_kernel)
denoised_dog = np.clip(denoised_dog, 0, 255)
print('Done!')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(dog_array, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original', fontsize=13)
axes[0].axis('off')

axes[1].imshow(noisy_dog, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Noisy (std=15)', fontsize=13)
axes[1].axis('off')

axes[2].imshow(denoised_dog, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('Denoised\n(7×7 Gaussian, σ=1.0)', fontsize=13)
axes[2].axis('off')

plt.suptitle('Question 3A: Gaussian Denoising', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('q3a_denoising.png', dpi=150, bbox_inches='tight')
plt.show()

# Compute noise metrics
mse_noisy    = np.mean((dog_array - noisy_dog)**2)
mse_denoised = np.mean((dog_array - denoised_dog)**2)
print(f'MSE (noisy vs original):    {mse_noisy:.2f}')
print(f'MSE (denoised vs original): {mse_denoised:.2f}')

In [ ]:
# --- Part B: Sharpening ---

sharpening_kernel = np.array([
    [1,  4,  6,  4, 1],
    [4, 16, 24, 16, 4],
    [6, 24, -476, 24, 6],
    [4, 16, 24, 16, 4],
    [1,  4,  6,  4, 1]
], dtype=np.float64) * -1.0 / 256.0

print('Sharpening kernel:')
print(sharpening_kernel)

print('\nApplying sharpening kernel...')
# Apply sharpening to the denoised image
sharpened_dog = convolve2d(denoised_dog, sharpening_kernel)
sharpened_dog = np.clip(sharpened_dog, 0, 255)
print('Done!')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(dog_array, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original', fontsize=13)
axes[0].axis('off')

axes[1].imshow(denoised_dog, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Denoised', fontsize=13)
axes[1].axis('off')

axes[2].imshow(sharpened_dog, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('Sharpened', fontsize=13)
axes[2].axis('off')

plt.suptitle('Question 3B: Image Sharpening via Convolution', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('q3b_sharpening.png', dpi=150, bbox_inches='tight')
plt.show()
print('Q3 figures saved.')

---
## Question 4: Template Matching – Convolution vs Correlation

We locate a product (`template.jpg`) inside a shelf image (`shelf.jpg`) using:
- **(a, b)** Template matching via **convolution** (flip template, subtract mean)
- **(c, d)** Template matching via **correlation** (no flip, subtract mean)
- **(e–g)** Compare and analyse both approaches

### Helper functions

In [ ]:
def correlate2d(image, template):
    """
    Perform 2D cross-correlation manually.
    Unlike convolution, the template is NOT flipped before sliding.
    
    Parameters:
        image    : np.ndarray (H, W)    - Grayscale image
        template : np.ndarray (kH, kW)  - Template (not flipped)
    Returns:
        output   : np.ndarray (H, W)    - Correlation map
    """
    img_h, img_w   = image.shape
    k_h, k_w       = template.shape
    pad_h, pad_w   = k_h // 2, k_w // 2

    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0)

    output = np.zeros((img_h, img_w), dtype=np.float64)
    for i in range(img_h):
        for j in range(img_w):
            region = padded[i:i + k_h, j:j + k_w]
            output[i, j] = np.sum(region * template)   # No flip
    return output


def subtract_mean(arr):
    """Subtract the mean from an array to reduce intensity bias."""
    return arr - arr.mean()


def find_best_match(response_map):
    """Return the (row, col) of the peak in a response map."""
    idx = np.unravel_index(np.argmax(response_map), response_map.shape)
    return idx

In [ ]:
# Load shelf and template images
shelf_img    = Image.open('shelf.jpg').convert('L')
template_img = Image.open('template.jpg').convert('L')

shelf    = np.array(shelf_img,    dtype=np.float64)
template = np.array(template_img, dtype=np.float64)

print(f'Shelf shape:    {shelf.shape}')
print(f'Template shape: {template.shape}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(shelf,    cmap='gray'); axes[0].set_title('Shelf Image');    axes[0].axis('off')
axes[1].imshow(template, cmap='gray'); axes[1].set_title('Product Template'); axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# --- (a, b) Template matching via CONVOLUTION ---
# Subtract mean from both image and template to reduce intensity bias
shelf_zm    = subtract_mean(shelf)     # zero-mean shelf
template_zm = subtract_mean(template)  # zero-mean template

# Flip the template before passing to convolve2d
# (convolve2d will flip it again internally, resulting in the original orientation)
template_flipped = np.flip(template_zm)   # pre-flip so convolve2d restores orientation

print('Running template matching via convolution...')
conv_response = convolve2d(shelf_zm, template_flipped)
print('Done!')

conv_match = find_best_match(conv_response)
print(f'Best match location (convolution): row={conv_match[0]}, col={conv_match[1]}')

In [ ]:
# --- (c, d) Template matching via CORRELATION ---
# Subtract mean; template is NOT flipped
print('Running template matching via correlation...')
corr_response = correlate2d(shelf_zm, template_zm)
print('Done!')

corr_match = find_best_match(corr_response)
print(f'Best match location (correlation): row={corr_match[0]}, col={corr_match[1]}')

In [ ]:
# --- Visualise response maps ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(conv_response, cmap='hot')
axes[0].plot(conv_match[1], conv_match[0], 'b+', markersize=15, markeredgewidth=2)
axes[0].set_title('Convolution Response Map', fontsize=13)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(corr_response, cmap='hot')
axes[1].plot(corr_match[1], corr_match[0], 'b+', markersize=15, markeredgewidth=2)
axes[1].set_title('Correlation Response Map', fontsize=13)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1])

plt.suptitle('Question 4: Template Matching Response Maps\n(blue + = best match)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q4_response_maps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Draw bounding boxes on shelf image ---
t_h, t_w = template.shape

def draw_box(ax, row, col, height, width, color, label):
    """Draw a rectangle bounding box on an axis."""
    from matplotlib.patches import Rectangle
    top_left_row = row - height // 2
    top_left_col = col - width  // 2
    rect = Rectangle((top_left_col, top_left_row), width, height,
                      linewidth=2, edgecolor=color, facecolor='none', label=label)
    ax.add_patch(rect)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Convolution result
axes[0].imshow(shelf, cmap='gray', vmin=0, vmax=255)
draw_box(axes[0], conv_match[0], conv_match[1], t_h, t_w, 'red', 'Convolution match')
axes[0].legend(loc='upper right', fontsize=10)
axes[0].set_title('Convolution: Product Location', fontsize=13)
axes[0].axis('off')

# Correlation result
axes[1].imshow(shelf, cmap='gray', vmin=0, vmax=255)
draw_box(axes[1], corr_match[0], corr_match[1], t_h, t_w, 'lime', 'Correlation match')
axes[1].legend(loc='upper right', fontsize=10)
axes[1].set_title('Correlation: Product Location', fontsize=13)
axes[1].axis('off')

plt.suptitle('Question 4: Template Matching – Located Product on Shelf', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('q4_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Q4 figures saved.')

---
## Question 4 (e–g): Analysis – Convolution vs Correlation for Template Matching

### (e) Which method more accurately locates the product?

In this template matching scenario, **both convolution and correlation produce the same result** when the template is pre-flipped before being passed to the convolution function. This is because:

- Correlation slides the template *as-is* over the image.
- Convolution flips the template 180° before sliding. If we pre-flip the template, the two operations become equivalent.

In practice, **correlation is more naturally suited** to template matching because it directly computes the similarity between the template and each image region without requiring a pre-flip step.

### (f) Which method is more efficient in computation and implementation?

| Aspect | Convolution | Correlation |
|---|---|---|
| Kernel flip | Required (adds a step) | Not required |
| Computation cost | Identical (same sliding window) | Identical |
| Implementation simplicity | Slightly more complex | Simpler |
| FFT applicability | Yes | Yes |

**Correlation is simpler to implement** for template matching: no pre-processing of the template is needed. Both methods have identical time complexity O(H·W·kH·kW) for a brute-force implementation, and both can be accelerated equally using FFT-based convolution.

### (g) Why one method may be better suited for template matching

**Correlation is the more natural and commonly used operation for template matching.** The fundamental goal is to measure how similar the template is to each region of the image. Correlation directly computes this similarity by placing the template at each position *in its original orientation* and summing element-wise products.

Convolution, by contrast, was designed for signal processing where the kernel is a response function (e.g., a filter), and the 180° flip is a mathematical convention. When used for template matching, you must remember to pre-flip the template to undo convolution's internal flip — an easy source of error.

Subtracting the mean from both the image patch and the template (as done in this assignment) is important because it removes the bias towards bright (high-intensity) regions. Without mean subtraction, the matcher would always prefer the brightest area of the image regardless of structural similarity to the template.

**Conclusion:** Use **correlation** for template matching. It is conceptually cleaner, requires fewer pre-processing steps, and is the standard approach in computer vision libraries (e.g., `cv2.matchTemplate`).